# data cleaning

In [ ]:
%pip install pandas numpy matplotlib seaborn scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
from sklearn.preprocessing import StandardScaler

# from thefuzz import process
from difflib import get_close_matches

In [129]:
train = pd.read_csv('train_(2)_(1)_(1)_(1).csv')
test = pd.read_csv('test_(2)_(1)_(1)_(1).csv')
avgRent = pd.read_csv('avg_rent_(1)_(1)_(1)_(1).csv')
disCityCenter = pd.read_csv('dist_from_city_centre_(1)_(1)_(1)_(1).csv')
sampleSubmission = pd.read_csv('sample_submission_(3)_(1)_(1)_(1).csv')

In [130]:
# print(train.info())
train.dropna(subset=['size', 'location', 'bath', 'balcony'], inplace=True)
# print(train.info())


### cleaning total_sqft and making it numeric 
as it is expected to be in sq feet

then scaling it

In [131]:
def is_number(x):
    try:
        float(x)
        return True
    except ValueError:
        return False

def return_sqft(x):
    if is_number(x):
        return float(x)
    
    tokens = x.split('-')
    if len(tokens) == 2:
        return (float(tokens[0]) + float(tokens[1])) / 2
    if x.endswith('Sq. Meter') or x.endswith('Sq. Meter'):
        x = float(x.split('Sq')[0])
        return x*10.7639
    if x.endswith('Acres'):
        x = float(x.split('Acr')[0])
        return x*43560
    if x.endswith('Sq. Yards'):
        x = float(x.split('Sq')[0])
        return x*10.7639
    if x.endswith('Grounds'):
        x = float(x.split('Gro')[0])
        return x*2400
    if x.endswith('Cents'):
        x = float(x.split('Cen')[0])
        return x*435.6
    if x.endswith('Perch'):
        x = float(x.split('Per')[0])
        return x*272.25
    if x.endswith('Guntha'):
        x = float(x.split('Gun')[0])
        return x*1089
    
    print("Couldn't parse the value: ", x)
    return x



train['total_sqft'] = train.total_sqft.apply(lambda x: return_sqft(x))
train[train.total_sqft.apply(lambda x: is_number(x) == False)]


train['price_per_sqft'] = train.price / train.total_sqft
# train.total_sqft.describe()

outlier treatment for total_sqft based on area_type


OUTLIERS HAVE NOT YET BEEN TREATED AND MUST BE TREATED FOR ALL CATEGORIES

In [132]:
# # FOR SOME REASON THE R^2 IS REDUCING
# # ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^ HENCE THE CODE IS COMMENTED OUT ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

# # TODO: outlier analysis to be done wrt area type 

# # plt.subplots = plt.subplots(figsize=(10,5))


# print(train.area_type.value_counts())
# plt.figure(figsize=(20,10))


# plt.subplot(2,4,1)
# plt.title('Price per sqft distribution for Super built-up  Area')
# sns.boxplot(train[train.area_type == 'Super built-up  Area'].price_per_sqft)
# plt.subplot(2,4,5)
# plt.title('Price per sqft distribution for Super built-up  Area')
# sns.histplot(train[train.area_type == 'Super built-up  Area'].price_per_sqft)



# plt.subplot(2,4,2)
# plt.title('Price per sqft distribution for Built-up  Area')
# sns.boxplot(train[train.area_type == 'Built-up  Area'].price_per_sqft)
# plt.subplot(2,4,6)
# plt.title('Price per sqft distribution for Built-up  Area')
# sns.histplot(train[train.area_type == 'Built-up  Area'].price_per_sqft)



# plt.subplot(2,4,3)
# plt.title('Price per sqft distribution for Plot  Area')
# sns.boxplot(train[train.area_type == 'Plot  Area'].price_per_sqft)
# plt.subplot(2,4,7)
# plt.title('Price per sqft distribution for Plot  Area')
# sns.histplot(train[train.area_type == 'Plot  Area'].price_per_sqft)



# plt.subplot(2,4,4)
# plt.title('Price per sqft distribution for Carpet  Area')
# sns.boxplot(train[train.area_type == 'Carpet  Area'].price_per_sqft)
# plt.subplot(2,4,8)
# plt.title('Price per sqft distribution for Carpet  Area')
# sns.histplot(train[train.area_type == 'Carpet  Area'].price_per_sqft)



# plt.tight_layout()
# plt.show()


for i in train.area_type.unique():
    col = train[train.area_type == i].price_per_sqft
    iqr = (col.quantile(0.75) - col.quantile(0.25)) * (3/2)
    train.drop(train[(train.area_type == i) & (train.price_per_sqft > iqr)].index, inplace=True)


In [133]:
# print(train.price_per_sqft.describe())
# print(train[train.price_per_sqft > .6])
# train.drop(train[train.price_per_sqft > 0.6].index, inplace=True, axis=0)


# scaling data => total sq feet


In [134]:
# note that we must do outlier treatment before scaling
# min max scaling for total_sqft

# train.total_sqft = train.total_sqft.apply(lambda x: (x - train.total_sqft.min())/(train.total_sqft.max() - train.total_sqft.min()))

successfully cleaned all non numeric data from total_sqft col

## handling size
- currently 1rk = 0
- 1bhk/1bedroom = 1
- 2bhk/2bedroom = 2 
and soo on

In [135]:
# print(train['size'].value_counts(dropna=False))
# train[train['size'].apply(lambda x: is_number(x) == False)]

def returnSize(x):
    try:
        if x == '1 RK':
            return 0
        else:
            return int(x.split(' ')[0])
    except:
        print("Couldn't parse size: ", x)


train['size'] = train['size'].apply(lambda x: returnSize(x))

# train.info()



## avilibility


In [136]:
from datetime import datetime

# train.availability.value_counts()

def noOfDays(x):
    if x == 'Ready To Move':
        return 0
    tokens = x.split('-')
    if len(tokens) == 2:
        now = datetime.now()

        date = datetime.strptime(f'{x}-{now.year}', '%d-%b-%Y')

        if (date - now).days < 0:
            date = datetime.strptime(f'{x}-{now.year + 1}', '%d-%b-%Y')
            
        return (date - now).days
    else:
        print("Couldn't parse availability: ", x)
        return -1


train.availability = train.availability.apply(lambda x: noOfDays(x))
# train.info()


## encoding area type

n-1 dummy encoding performed 

with value counts

- area_type
- Super built-up  Area    6745
- Built-up  Area          1845
- Plot  Area              1496
- Carpet  Area              65

model performance to be tested including only Super built-up  Area 

In [137]:
temp = True
for i in train.area_type.unique():
    if temp:
        temp = False
        continue
    train["area_type_" + i.split("  ")[0].replace(" ", "_").replace("-", "_")] = train.area_type.apply(lambda x: 1 if x==i else 0)

train.drop('area_type', axis=1, inplace=True)

In [138]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2122 entries, 1 to 10651
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        2122 non-null   int64  
 1   availability              2122 non-null   int64  
 2   location                  2122 non-null   object 
 3   size                      2122 non-null   int64  
 4   society                   831 non-null    object 
 5   total_sqft                2122 non-null   float64
 6   bath                      2122 non-null   float64
 7   balcony                   2122 non-null   float64
 8   price                     2122 non-null   float64
 9   price_per_sqft            2122 non-null   float64
 10  area_type_Super_built_up  2122 non-null   int64  
 11  area_type_Built_up        2122 non-null   int64  
 12  area_type_Carpet          2122 non-null   int64  
dtypes: float64(5), int64(6), object(2)
memory usage: 232.1+ KB


# using average rent and distance from city center

In [ ]:

# # Function to find the best match and its score using thefuzz.process.extractOne
# def find_closest_match(target, choices):
#     # process.extractOne returns a tuple: (closest_match, score, index)
#     match, score, _ = process.extractOne(target, choices)
#     return match, score

# # Apply the function across rows
# # The result will be a new DataFrame or Series containing the match and score
# train[['avgRentPlace', 'avgRentScore']] = train['location'].apply(
#     lambda x: pd.Series(find_closest_match(x, avgRent['location']))
# )


# train['average_rent'] = train.apply(
#     lambda row: avgRent.loc[avgRent['location'] == row['avgRentPlace'], 'avg_2bhk_rent'].values[0],
#     axis=1
# )

# # train.drop(['avgRentPlace', 'avgRentScore'], axis=1, inplace=True)


# train[['distanceField', 'distanceScore']] = train['location'].apply(
#     lambda x: pd.Series(find_closest_match(x, disCityCenter['location']))
# )

# train["distance_from_city_center"] = train.apply(
#     lambda row: disCityCenter.loc[
#         disCityCenter["location"] == row["distanceField"], "dist_from_city"
#     ].values[0],
#     axis=1,
# )

# # train.drop(['distanceField', 'distanceScore'], axis=1, inplace=True)


In [141]:
train.head(20)

,ID,availability,location,size,society,total_sqft,bath,balcony,price,price_per_sqft,area_type_Super_built_up,area_type_Built_up,area_type_Carpet,avgRentPlace,avgRentScore,average_rent,distanceField,distanceScore,distance_from_city_center
1,1,0,Chikka Tirupathi,4,Theanmp,2600.000000,5.000000,3.000000,120.000000,0.046154,0,0,0,Panathur,55,16000,Chikka Tirupathi,100,34.600000
5,5,0,Whitefield,2,DuenaTa,1170.000000,2.000000,1.000000,38.000000,0.032479,1,0,0,Whitefield,100,14981,Whitefield,100,17.300000
11,11,0,Whitefield,4,Prrry M,2785.000000,5.000000,3.000000,295.000000,0.105925,0,0,0,Whitefield,100,14981,Whitefield,100,17.300000
13,13,0,Gottigere,2,NaN,1100.000000,2.000000,2.000000,40.000000,0.036364,0,1,0,Gottigere,100,11000,Gottigere,100,15.200000
14,14,0,Sarjapur,3,Skityer,2250.000000,3.000000,2.000000,148.000000,0.065778,0,0,0,Sarjapur,100,45000,Sarjapur,100,17.200000
20,20,0,Kengeri,1,NaN,600.000000,1.000000,1.000000,15.000000,0.025000,0,1,0,Kengeri,100,12000,Kengeri,100,22.500000
26,26,0,Electronic City,2,Itelaa,660.000000,1.000000,1.000000,23.100000,0.035000,1,0,0,Electronics City,97,10650,Electronic City,100,18.100000
31,31,0,Bisuvanahalli,3,Prityel,1075.000000,2.000000,1.000000,35.000000,0.032558,1,0,0,Kasavanahalli,77,14808,Bisuvanahalli,100,38.400000
33,33,0,Raja Rajeshwari Nagar,3,GrrvaGr,1693.000000,3.000000,3.000000,57.390000,0.033898,1,0,0,J. P. Nagar,86,17600,Raja Rajeshwari Nagar,100,15.300000
41,41,337,Sarjapur Road,3,Soini T,1254.000000,3.000000,2.000000,38.000000,0.030303,1,0,0,Sarjapur Road,96,19000,Sarjapur Road,100,17.200000


# standard scaling

In [142]:
scaler = StandardScaler()

numeric_cols = train.select_dtypes(include=[np.number]).columns
train[numeric_cols] = scaler.fit_transform(train[numeric_cols])

In [143]:
# import 'Pandas' 
import pandas as pd 

# import 'Numpy' 
import numpy as np

# import subpackage of Matplotlib
import matplotlib.pyplot as plt

# import 'Seaborn' 
import seaborn as sns

# to suppress warnings 
from warnings import filterwarnings
filterwarnings('ignore')

# display all columns of the dataframe
pd.options.display.max_columns = None

# display all rows of the dataframe
pd.options.display.max_rows = None
 
# to display the float values upto 6 decimal places     
pd.options.display.float_format = '{:.6f}'.format

# import train-test split 
from sklearn.model_selection import train_test_split

# # import various functions from statsmodels
# import statsmodels
import statsmodels.api as sm
# import statsmodels.stats.api as sms
# from statsmodels.graphics.gofplots import qqplot

# # import 'stats'
# from scipy import stats

# # 'metrics' from sklearn is used for evaluating the model performance
# from sklearn.metrics import mean_squared_error

# # import functions to perform feature selection
# from mlxtend.feature_selection import SequentialFeatureSelector as sfs
# #from sklearn.feature_selection import SelectFromModel
from sklearn.feature_selection import RFE

# import function to perform linear regression
from sklearn.linear_model import LinearRegression

# import functions to perform cross validation
from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [144]:
train.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2122 entries, 1 to 10651
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ID                         2122 non-null   float64
 1   availability               2122 non-null   float64
 2   location                   2122 non-null   object 
 3   size                       2122 non-null   float64
 4   society                    831 non-null    object 
 5   total_sqft                 2122 non-null   float64
 6   bath                       2122 non-null   float64
 7   balcony                    2122 non-null   float64
 8   price                      2122 non-null   float64
 9   price_per_sqft             2122 non-null   float64
 10  area_type_Super_built_up   2122 non-null   float64
 11  area_type_Built_up         2122 non-null   float64
 12  area_type_Carpet           2122 non-null   float64
 13  avgRentPlace               2122 non-null   object 
 

In [145]:
test.head()

,ID,area_type,availability,location,size,society,total_sqft,bath,balcony
0,0,Super built-up Area,Ready To Move,Chamrajpet,2 BHK,NaN,650,1.000000,1.000000
1,1,Super built-up Area,Ready To Move,7th Phase JP Nagar,3 BHK,SrncyRe,1370,2.000000,1.000000
2,2,Super built-up Area,Ready To Move,Whitefield,3 BHK,AjhalNa,1725,3.000000,2.000000
3,3,Built-up Area,Ready To Move,Jalahalli,2 BHK,NaN,1000,2.000000,0.000000
4,4,Plot Area,Ready To Move,TC Palaya,1 Bedroom,NaN,1350,1.000000,0.000000


In [146]:
train.drop(['price_per_sqft'], axis=1, inplace=True)

x = train.drop('price', axis=1)
y = train['price']

x = x.select_dtypes(include= 'number')
x = sm.add_constant(x)

X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=1, test_size = 0.2)

# xTest = test.drop('price', axis=1)
# yTest = test['price']

In [147]:
myModel = sm.OLS(y_train, X_train).fit()
myModel.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                  price   R-squared:                       0.394
Model:                            OLS   Adj. R-squared:                  0.389
Method:                 Least Squares   F-statistic:                     84.04
Date:                Thu, 15 Jan 2026   Prob (F-statistic):          3.96e-172
Time:                        23:43:47   Log-Likelihood:                -1960.2
No. Observations:                1697   AIC:                             3948.
Df Residuals:                    1683   BIC:                             4025.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
=============================================================================================
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
const                        -0.0044      0.019     -0.233      0.816      -0.041       0.032
ID                            0.0080      0.019      0.433      0.665      -0.028       0.044
availability                 -0.0368      0.019     -1.937      0.053      -0.074       0.000
size                         -0.1112      0.043     -2.593      0.010      -0.195      -0.027
total_sqft                    0.0143      0.018      0.779      0.436      -0.022       0.050
bath                          0.3525      0.044      8.059      0.000       0.267       0.438
balcony                       0.1584      0.020      7.863      0.000       0.119       0.198
area_type_Super_built_up     -0.4461      0.023    -19.054      0.000      -0.492      -0.400
area_type_Built_up           -0.3067      0.021    -14.290      0.000      -0.349      -0.265
area_type_Carpet             -0.0727      0.019     -3.880      0.000      -0.109      -0.036
avgRentScore                  0.0641      0.020      3.191      0.001       0.025       0.103
average_rent                 -0.0022      0.017     -0.128      0.898      -0.036       0.032
distanceScore                 0.0107      0.020      0.530      0.596      -0.029       0.050
distance_from_city_center    -0.0080      0.019     -0.416      0.677      -0.046       0.030
==============================================================================
Omnibus:                     1867.613   Durbin-Watson:                   1.920
Prob(Omnibus):                  0.000   Jarque-Bera (JB):           252648.108
Skew:                           5.246   Prob(JB):                         0.00
Kurtosis:                      61.848   Cond. No.                         4.86
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [148]:
# initiate linear regression model to use in feature selection
linreg = LinearRegression()

# build forward feature selection
# pass the regression model to 'estimator'
# pass number of required feartures to 'k_features'. Here '12' is the stopping rule
# 'forward=True' performs forward selection method
# 'verbose=1' returns the number of features at the corresponding step
# 'verbose=2' returns the R-squared scores and the number of features at the corresponding step
# 'scoring=r2' considers R-squared score to select the feature
# linreg_forward = sfs(estimator=linreg, k_features = 12, forward=True,
#                      verbose=2, scoring='r2')

# fit the forward selection on training data using fit()
# sfs_forward = linreg_forward.fit(X_train, y_train)


linreg.fit(X_train, y_train)

print(f"Model Score: {model.score(X_test, y_test)}")

NameError: name 'model' is not defined